# Before You Start

This notebook runs the Civic Connect AI service temporarily in Google Colab for a live mobile demo.

## Required setup

1. Open **Runtime > Change runtime type** and select Python 3.
2. Use a CPU runtime or a T4 GPU runtime if one is available.
3. Run every code cell from top to bottom.
4. Wait until the final cell prints `classifier: ready` and an `AI_SERVICE_URL`.
5. Open the printed `/health` URL and confirm it returns HTTP 200.
6. In Render, open the `civic-connect-api` service and set `AI_SERVICE_URL` to the printed tunnel URL. Do not add `/health` or `/api`.
7. Redeploy the Node API after changing the environment variable.
8. Keep this Colab tab and runtime running while using the Flutter app outside your laptop.

The phone continues calling the Render API through `API_BASE_URL`. Render calls this temporary Colab service through Cloudflare Tunnel. The tunnel URL changes if Colab restarts, so update Render and redeploy the Node API whenever that happens.

This setup is for demonstrations only. Colab may disconnect, reclaim the runtime, or require the model to download again.

# Civic Connect AI Demo

This notebook starts the FastAPI vision service in Google Colab and exposes it through a temporary Cloudflare HTTPS tunnel.

Keep this tab open while demonstrating the mobile app. Copy the printed tunnel URL into the Render Node API environment variable `AI_SERVICE_URL`.

## Important demo limits

- Colab is temporary and may disconnect.
- The tunnel URL changes whenever the runtime restarts.
- The phone does not need to be on the same Wi-Fi network.
- Do not use this setup for production or sensitive data.

In [1]:
# Clone the repository into the Colab runtime.
# Change REPO_URL if the repository is private or has moved.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Varunpatel586/Civic_Connect.git'
WORKSPACE = Path('/content/Civic_Connect')

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKSPACE)], check=True)
AI_DIR = WORKSPACE / 'ai_service'
os.chdir(AI_DIR)
print(f'AI service directory: {AI_DIR}')

AI service directory: /content/Civic_Connect/ai_service


In [2]:
# Install a Colab-compatible AI stack.
# The repository pins are used for server deployments, but Colab's Python 3.13
# runtime needs matching Torch and torchvision wheels for Transformers pipeline.
subprocess.run([
    'python', '-m', 'pip', 'install', '-q',
    'fastapi==0.141.1',
    'uvicorn==0.52.4',
    'pydantic==2.13.5',
    'python-multipart==0.0.32',
    'opencv-python-headless==4.10.0.84',
    'numpy==2.1.3',
    'Pillow==11.0.0',
    'torch==2.6.0',
    'torchvision==0.21.0',
    'transformers==4.48.3',
], check=True)

# Verify the import that previously caused Uvicorn to exit.
subprocess.run([
    'python', '-c',
    'from transformers import pipeline; import torch, torchvision; '
    "print(f'pipeline import OK; torch={torch.__version__}; torchvision={torchvision.__version__}')",
], check=True)
print('Colab-compatible dependencies installed.')

Colab-compatible dependencies installed.


In [3]:
# Pre-download and initialize CLIP before starting Uvicorn.
# This makes Hugging Face progress visible and leaves model files in the local cache.
import torch
from PIL import Image
from transformers import pipeline

print('Pre-loading CLIP model into cache...')
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (CUDA)' if device == 0 else 'CPU'}")

classifier = pipeline(
    'zero-shot-image-classification',
    model='openai/clip-vit-base-patch32',
    device=device,
)

# Use an in-memory RGB image so the smoke test does not depend on a sample URL.
test_image = Image.new('RGB', (224, 224), color=(128, 128, 128))
classifier(test_image, candidate_labels=['cat', 'dog'])
print('CLIP model successfully loaded and cached!')

Pre-loading CLIP model into cache...
Using device: CPU


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


CLIP model successfully loaded and cached!


In [4]:
# Download cloudflared for the Colab Linux runtime.
CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(CLOUDFLARED),
    ], check=True)
    CLOUDFLARED.chmod(0o755)
print(f'cloudflared ready: {CLOUDFLARED}')

cloudflared ready: /content/cloudflared


In [5]:
# Start FastAPI and wait until the pre-warmed CLIP service is ready.
import json
import time
import urllib.error
import urllib.request

API_LOG = Path('/content/fastapi.log')
if API_LOG.exists():
    API_LOG.unlink()

API_LOG_HANDLE = API_LOG.open('w')
API_PROCESS = subprocess.Popen([
    'uvicorn', 'main:app',
    '--host', '0.0.0.0',
    '--port', '8000',
], cwd=AI_DIR, stdout=API_LOG_HANDLE, stderr=subprocess.STDOUT, text=True)

ready = False
deadline = time.time() + 90  # Pre-warm should make startup quick.
last_health = None
while time.time() < deadline:
    if API_PROCESS.poll() is not None:
        API_LOG_HANDLE.flush()
        log_tail = API_LOG.read_text(errors='ignore')
        raise RuntimeError(
            f'FastAPI process crashed on startup:\n\n{log_tail}'
        )

    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3) as response:
            last_health = json.loads(response.read().decode())
            print('Health check response:', last_health)
            if (
                last_health.get('classifier') == 'ready'
                or last_health.get('status') == 'ok'
            ):
                ready = True
                break
    except urllib.error.URLError:
        pass

    time.sleep(3)

API_LOG_HANDLE.flush()
if not ready:
    log_tail = API_LOG.read_text(errors='ignore')
    API_PROCESS.terminate()
    raise RuntimeError(
        f'FastAPI failed to start within 90 seconds. Logs:\n\n{log_tail}'
    )

API_LOG_HANDLE.close()
print('FastAPI service is running and ready for requests.')

RuntimeError: FastAPI failed to start within 90 seconds. Logs:

INFO:     Started server process [4363]
INFO:     Waiting for application startup.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:51236 - "GET /health HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:48280 - "GET /health HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:48290 - "GET /health HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:48298 - "GET /health HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:41084 - "GET /health HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:41088 - "GET /health HTTP/1.1" 404 Not Found
INFO:

In [ ]:
# Create the temporary public HTTPS tunnel.
TUNNEL_LOG = Path('/content/cloudflared.log')
TUNNEL_PROCESS = subprocess.Popen([
    str(CLOUDFLARED), 'tunnel',
    '--url', 'http://127.0.0.1:8000',
    '--no-autoupdate',
    '--logfile', str(TUNNEL_LOG),
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

tunnel_url = None
deadline = time.time() + 60
while time.time() < deadline:
    if TUNNEL_LOG.exists():
        log = TUNNEL_LOG.read_text(errors='ignore')
        for token in log.split():
            if token.startswith('https://') and 'trycloudflare.com' in token:
                tunnel_url = token.rstrip('\"\,')
                break
    if tunnel_url:
        break
    time.sleep(2)

if not tunnel_url:
    raise RuntimeError('Cloudflare did not provide a tunnel URL. Check /content/cloudflared.log.')

print('AI_SERVICE_URL = ' + tunnel_url)
print('Health check: ' + tunnel_url + '/health')
print('Set this URL on the Render Node API, then redeploy the Node service.')

## Mobile demo checklist

1. Confirm the tunnel URL returns `classifier: ready` at `/health`.
2. In Render, set `AI_SERVICE_URL` to the printed URL without `/health`.
3. Redeploy the Node API.
4. Fully restart the Flutter app if its API configuration changed.
5. Take a pothole photograph from the phone.
6. Leave this Colab runtime running during the demo.

The Flutter app continues using the Render API URL. It should not call the Colab URL directly.